# Modelado Predictivo de Riesgo Crediticio
**Metodología:** Regresión Logística (baseline) · Random Forest · XGBoost (campeón)  
**Framework regulatorio de referencia:** Basilea III / IRB (Internal Ratings-Based Approach)  
**Objetivo:** Estimar la Probabilidad de Default (PD) por cliente y segmentar el portafolio por nivel de riesgo.

In [2]:
# =============================================================================
# 0. IMPORTACIONES
# =============================================================================
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    auc, classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

ModuleNotFoundError: No module named 'numpy'

## 1. Ingesta y preparación de datos

In [ ]:
# =============================================================================
# 1. INGESTA
# =============================================================================
df = pd.read_csv('../datos/02_datos_para_powerbi.csv')
print(f"Shape: {df.shape}")
print(f"Default rate: {df['loan_status'].mean():.2%}")
df.head(3)

## 2. Ingeniería de características

In [ ]:
# =============================================================================
# 2. FEATURE ENGINEERING
# =============================================================================
CATEGORICAS = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']
NUMERICAS   = ['person_age', 'person_income', 'person_emp_length',
                'loan_amnt', 'loan_int_rate', 'loan_percent_income',
                'cb_person_cred_hist_length']

y = df['loan_status']
X = df.drop('loan_status', axis=1)

# One-Hot Encoding con drop_first para evitar multicolinealidad perfecta
X_enc = pd.get_dummies(X, columns=CATEGORICAS, drop_first=True)

## 3. Partición estratificada y escalamiento

In [ ]:
# =============================================================================
# 3. TRAIN / TEST SPLIT (estratificado)
# =============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y,
    test_size=0.25,
    stratify=y,
    random_state=42,
)
print(f"Train: {X_train.shape} | Default rate: {y_train.mean():.2%}")
print(f"Test:  {X_test.shape}  | Default rate: {y_test.mean():.2%}")

# =============================================================================
# 4. ESCALAMIENTO Z-SCORE (fit en train, transform en ambos)
# =============================================================================
scaler = StandardScaler()
X_train[NUMERICAS] = scaler.fit_transform(X_train[NUMERICAS])
X_test[NUMERICAS]  = scaler.transform(X_test[NUMERICAS])
print("Escalamiento Z-score aplicado sin fuga de datos.")

## 4. Modelo baseline — Regresión Logística

Referencia metodológica bajo el enfoque IRB de Basilea III. 
Interpretable, auditable y útil para el análisis de coeficientes / odds ratios.

In [ ]:
# =============================================================================
# 5. ENTRENAMIENTO
# =============================================================================
log_reg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

# Coeficientes e interpretación económica
df_coef = pd.DataFrame({
    'Variable':    X_train.columns,
    'Beta':        log_reg.coef_[0],
    'Odds Ratio':  np.exp(log_reg.coef_[0]),
}).sort_values('Odds Ratio', ascending=False)

print("Top 5 predictores por Odds Ratio:")
print(df_coef.head(5).to_string(index=False))

In [ ]:
# =============================================================================
# 6. EVALUACIÓN — umbral estándar (0.50)
# =============================================================================
y_pred_log  = log_reg.predict(X_test)
y_proba_log = log_reg.predict_proba(X_test)[:, 1]

auc_log  = roc_auc_score(y_test, y_proba_log)
gini_log = 2 * auc_log - 1

print(classification_report(y_test, y_pred_log, target_names=['Al dia', 'Default']))
print(f"AUC-ROC : {auc_log:.4f}")
print(f"Gini    : {gini_log:.4f}")

In [ ]:
# =============================================================================
# 7. EVALUACION VISUAL — 3 paneles
# =============================================================================
UMBRAL_NEGOCIO = 0.25
y_pred_opt = (y_proba_log >= UMBRAL_NEGOCIO).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Confusion matrix (umbral de negocio)
cm = confusion_matrix(y_test, y_pred_opt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False, ax=axes[0],
            xticklabels=['Al dia', 'Default'], yticklabels=['Al dia', 'Default'],
            annot_kws={'size': 14, 'weight': 'bold'})
axes[0].set_title(f'Confusion Matrix (umbral={UMBRAL_NEGOCIO})', weight='bold')
axes[0].set_xlabel('Prediccion')
axes[0].set_ylabel('Real')

# Panel B: Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba_log)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC={auc_log:.4f}')
axes[1].plot([0, 1], [0, 1], 'navy', lw=1, linestyle='--')
axes[1].fill_between(fpr, tpr, alpha=0.08, color='darkorange')
axes[1].set(xlabel='FPR (1-Especificidad)', ylabel='TPR (Recall)',
             title='Curva ROC')
axes[1].legend()

# Panel C: Curva de calibracion
prob_true, prob_pred = calibration_curve(y_test, y_proba_log, n_bins=10, strategy='uniform')
axes[2].plot(prob_pred, prob_true, 'o-', color='forestgreen', label='Logit')
axes[2].plot([0, 1], [0, 1], '--', color='gray', label='Perfecta')
axes[2].set(xlabel='Probabilidad predicha', ylabel='Proporcion real de defaults',
             title='Curva de Calibracion')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 8. SERIALIZACION
# =============================================================================
with open('modelo_logistica_riesgo.pkl', 'wb') as f:
    pickle.dump(log_reg, f)
with open('escalador_standard.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Modelo y escalador serializados correctamente.")

In [ ]:
# =============================================================================
# 9. EXPORTACION — dataset predictivo para Power BI (Logit)
# =============================================================================
df_test_pbi = X_test.copy()
df_test_pbi[NUMERICAS] = scaler.inverse_transform(X_test[NUMERICAS])
df_test_pbi['probability_of_default (PD)'] = y_proba_log
df_test_pbi['predicted_status_u025']        = y_pred_opt
df_test_pbi['real_status (Target)']         = y_test
df_test_pbi.to_csv('../datos/03_resultados_predicciones_test.csv', index=False)
print("03_resultados_predicciones_test.csv exportado.")

## 5. Modelos de ensamble — Random Forest y XGBoost

In [ ]:
# =============================================================================
# 10. RANDOM FOREST (Bagging)
# =============================================================================
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',   # compensa el desbalance de clases
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)

y_pred_rf  = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

auc_rf = roc_auc_score(y_test, y_proba_rf)
print(f"Random Forest — AUC: {auc_rf:.4f} | Gini: {2*auc_rf-1:.4f}")
print(classification_report(y_test, y_pred_rf, target_names=['Al dia', 'Default']))

In [ ]:
# =============================================================================
# 11. XGBOOST (Boosting) — modelo campeon
# =============================================================================
# scale_pos_weight = n_negativos / n_positivos penaliza los falsos negativos
ratio_desbalance = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=ratio_desbalance,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train)

y_pred_xgb  = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

auc_xgb = roc_auc_score(y_test, y_proba_xgb)
print(f"XGBoost — AUC: {auc_xgb:.4f} | Gini: {2*auc_xgb-1:.4f}")
print(classification_report(y_test, y_pred_xgb, target_names=['Al dia', 'Default']))

## 6. Interpretabilidad (XAI) y comparativa de modelos

In [ ]:
# =============================================================================
# 12. FEATURE IMPORTANCE — XGBoost (top 10)
# =============================================================================
importancias = (
    pd.Series(xgb_model.feature_importances_, index=X_train.columns)
    .sort_values()
    .tail(10)
)

fig, ax = plt.subplots(figsize=(10, 6))
importancias.plot(kind='barh', color='#10B981', ax=ax)
ax.set_title('Top 10 variables — XGBoost (reduccion de impureza Gini)', weight='bold')
ax.set_xlabel('Importancia relativa')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 13. CURVAS ROC COMPARATIVAS — Logit vs. XGBoost
# =============================================================================
fpr_log, tpr_log, _ = roc_curve(y_test, y_proba_log)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_proba_xgb)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_log, tpr_log, '--', color='navy',       label=f'Logit baseline (AUC={auc_log:.4f})')
ax.plot(fpr_xgb, tpr_xgb,       color='darkorange', label=f'XGBoost campeon (AUC={auc_xgb:.4f})', lw=2)
ax.plot([0, 1], [0, 1], ':', color='gray')
ax.set(xlabel='FPR (1-Especificidad)', ylabel='TPR (Sensibilidad)',
       title='Comparativa ROC: Logit vs. XGBoost')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Exportación final — dataset para Power BI

Se reconstruye el set de test con valores originales (sin escalar), 
se inyectan las métricas del modelo campeón y se segmenta el portafolio en tres bandas de riesgo.

In [ ]:
# =============================================================================
# 14. EXPORTACION — dataset predictivo campeon (XGBoost) para Power BI
# =============================================================================

# Reconstruimos con valores originales del DataFrame fuente
df_final = df.loc[X_test.index].copy()

# Metricas del modelo campeon
df_final['probability_of_default_xgb (%)'] = y_proba_xgb * 100
df_final['predicted_status_xgb']            = y_pred_xgb
df_final['real_status (Target)']            = y_test

# Segmentacion de riesgo (capa de negocio)
bins    = [0, 20, 50, 100]
labels  = ['Riesgo Bajo', 'Riesgo Medio', 'Riesgo Alto']
df_final['segmento_riesgo_IA'] = pd.cut(
    df_final['probability_of_default_xgb (%)'],
    bins=bins, labels=labels, right=False,
)

# Serializacion del modelo campeon
with open('../datos/modelo_xgboost_riesgo.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

df_final.to_csv('../datos/04_resultados_scoring_xgboost.csv', index=False)

print("Archivos exportados:")
print("  modelo_xgboost_riesgo.pkl")
print("  04_resultados_scoring_xgboost.csv")
print(f"\nDistribucion de segmentos:")
print(df_final['segmento_riesgo_IA'].value_counts())